In [73]:
#pip install --upgrade --force-reinstall  git+https://github.com/kanawa12/symcontroltools

In [74]:
#pip install git+https://github.com/kanawa12/symcontroltools

import numpy as np
import sympy as sp
import control as ct
import symcontroltools as sct
from symcontroltools import dp
import knw_symtools as ks

シンボリック変数を生成

In [75]:

# -----
ndim = 2; adim = 2; rdim = 2
pin  = 1; pout = 1; rin  = 1; rout = 1
ddict = dict([("NdimRdim", (ndim, rdim))])
#プラント,プラント入出力,規範入出力の次元
# -----システムの行列のシンボリック変数の定義-----
zm = sp.Matrix([[0]]) # 1次元の零行列 こうしないとサイズエラー
(ap, bp, cp), (Ap, Bp, Cp, Dp) = sct.get_SISO_sims(adim, "p")
(am, bm, cm), (Am, Bm, Cm, Dm) = sct.get_SISO_sims(rdim, "m")

# -----CGTの係数?の定義-----
SA = sp.Matrix( sa := sp.symbols(sct.setf("s", "a", rdim*adim)) ).reshape(adim, rdim)
SB = sp.Matrix( sb := sp.symbols(sct.setf("s", "b", adim)) ).reshape(adim, 1)
# -----SACゲインの定義-----
Kx = sp.Matrix( kx := sp.symbols(sct.setf("k", "x", rdim)) ).reshape(1,rdim)
Ku = sp.Matrix( ku := sp.symbols(sct.setf("k", "u", rin)) ).reshape(1,rin)
print(sp.srepr(sp.Tuple(kx, ku)))
dp("規範モデル", pmode=2)
dp(refmodel := sp.Tuple(Am, Bm, Cm))
dp("プラント", pmode=2)
dp(plant := sp.Tuple(Ap, Bp, Cp))
dp("?", pmode=2)
dp(cgtsyms := sp.Tuple(SA, SB, Kx, Ku))
knowns = None; unknowns = None
alpha, delta = sp.symbols(r"\alpha \delta")
#display(alpha+delta)
ddict.update([("refmodel", sp.srepr(refmodel)), ("plant", plant), ("cgtsyms", cgtsyms)])


Ap = Ap.subs([(ap[1], 0), (ap[2], 0), ]);dp(Ap)


Am = Am.subs([(am[1], 0), (am[2], 0) ])

#Bp = Bp.subs([(bp[0], 1), (bp[1], 1), (bp[2], 1)])
#Bm = Bm.subs([(bm[0], 1), (bm[1], 1), (bm[2], 1)])

Cp = Cp.subs([(cp[0], 1), (cp[1], 1)])
Cm = Cm.subs([(cm[0], 1), (cm[1], 1)])
s = sp.symbols("s")
#dp(sp.simplify(sct.get_tf(Ap, Bp, Cp, s)))


Tuple(Tuple(Symbol('{k_{x}}_{1}'), Symbol('{k_{x}}_{2}')), Tuple(Symbol('{k_{u}}_{1}')))


'規範モデル'

(Matrix([
[{a_{m}}_{1}, {a_{m}}_{2}],
[{a_{m}}_{3}, {a_{m}}_{4}]]), Matrix([
[{b_{m}}_{1}],
[{b_{m}}_{2}]]), Matrix([[{c_{m}}_{1}, {c_{m}}_{2}]]))

'プラント'

(Matrix([
[{a_{p}}_{1}, {a_{p}}_{2}],
[{a_{p}}_{3}, {a_{p}}_{4}]]), Matrix([
[{b_{p}}_{1}],
[{b_{p}}_{2}]]), Matrix([[{c_{p}}_{1}, {c_{p}}_{2}]]))

'?'

(Matrix([
[{s_{a}}_{1}, {s_{a}}_{2}],
[{s_{a}}_{3}, {s_{a}}_{4}]]), Matrix([
[{s_{b}}_{1}],
[{s_{b}}_{2}]]), Matrix([[{k_{x}}_{1}, {k_{x}}_{2}]]), Matrix([[{k_{u}}_{1}]]))

Matrix([
[{a_{p}}_{1},           0],
[          0, {a_{p}}_{4}]])

In [76]:

Aps = Ap; Bps = Bp; Cps = Cp
Aps = sp.Matrix([ [Aps, Bps], [Cps, zm] ])
Amm = sp.Matrix([ [Am , Bm],  [Cm , zm] ])
SKs = sp.Matrix([[SA, SB],[Kx, Ku]])

dp(rt := sp.Tuple(Aps, Amm, SKs))
Ams = sp.Matrix([[SA * Am, SA*Bm], [Cm, zm]])
ddict.update([("cgt", rt)])

exprr = Aps*SKs - Ams
#dp(sp.Matrix([Aps, SKs]))
dp("Aps, Amm, SKs", pmode=2)
dp(Ams, "Ams", pmode=5); dp(exprr, "$Aps*SKs - Ams=$", pmode=5)

(Matrix([
[{a_{p}}_{1},           0, {b_{p}}_{1}],
[          0, {a_{p}}_{4}, {b_{p}}_{2}],
[          1,           1,           0]]), Matrix([
[{a_{m}}_{1},           0, {b_{m}}_{1}],
[          0, {a_{m}}_{4}, {b_{m}}_{2}],
[          1,           1,           0]]), Matrix([
[{s_{a}}_{1}, {s_{a}}_{2}, {s_{b}}_{1}],
[{s_{a}}_{3}, {s_{a}}_{4}, {s_{b}}_{2}],
[{k_{x}}_{1}, {k_{x}}_{2}, {k_{u}}_{1}]]))

'Aps, Amm, SKs'

'Ams'

Matrix([
[{a_{m}}_{1}*{s_{a}}_{1}, {a_{m}}_{4}*{s_{a}}_{2}, {b_{m}}_{1}*{s_{a}}_{1} + {b_{m}}_{2}*{s_{a}}_{2}],
[{a_{m}}_{1}*{s_{a}}_{3}, {a_{m}}_{4}*{s_{a}}_{4}, {b_{m}}_{1}*{s_{a}}_{3} + {b_{m}}_{2}*{s_{a}}_{4}],
[                      1,                       1,                                                 0]])

'$Aps*SKs - Ams=$'

Matrix([
[-{a_{m}}_{1}*{s_{a}}_{1} + {a_{p}}_{1}*{s_{a}}_{1} + {b_{p}}_{1}*{k_{x}}_{1}, -{a_{m}}_{4}*{s_{a}}_{2} + {a_{p}}_{1}*{s_{a}}_{2} + {b_{p}}_{1}*{k_{x}}_{2}, {a_{p}}_{1}*{s_{b}}_{1} - {b_{m}}_{1}*{s_{a}}_{1} - {b_{m}}_{2}*{s_{a}}_{2} + {b_{p}}_{1}*{k_{u}}_{1}],
[-{a_{m}}_{1}*{s_{a}}_{3} + {a_{p}}_{4}*{s_{a}}_{3} + {b_{p}}_{2}*{k_{x}}_{1}, -{a_{m}}_{4}*{s_{a}}_{4} + {a_{p}}_{4}*{s_{a}}_{4} + {b_{p}}_{2}*{k_{x}}_{2}, {a_{p}}_{4}*{s_{b}}_{2} - {b_{m}}_{1}*{s_{a}}_{3} - {b_{m}}_{2}*{s_{a}}_{4} + {b_{p}}_{2}*{k_{u}}_{1}],
[                                               {s_{a}}_{1} + {s_{a}}_{3} - 1,                                                {s_{a}}_{2} + {s_{a}}_{4} - 1,                                                                             {s_{b}}_{1} + {s_{b}}_{2}]])

In [77]:
dp(vars := [*sa,*sb])

[{s_{a}}_{1}, {s_{a}}_{2}, {s_{a}}_{3}, {s_{a}}_{4}, {s_{b}}_{1}, {s_{b}}_{2}]

In [78]:
es = exprr.shape
print(es)
dp(expr := exprr.reshape(es[0]*es[1], 1))

ssmat, ssmaty = sp.linear_eq_to_matrix(expr, vars)
dp(sp.Tuple(ssmat,ssmaty))
E, ssdict = ks.solve_S(sp.Matrix([[ssmat, ssmaty]]), vars)
dp(E); dp(ssdict)

expr = expr.subs( ssdict ); dp(expr)

dp(expr := sp.simplify(sp.expand(expr)))
dp(expr := ks.elim_zero_row(expr))

kvars = [*kx, *ku]; dp(kvars)
Ek, sskdict = ks.solve_S(sp.Matrix([[*sp.linear_eq_to_matrix(expr, kvars)]]), kvars)
dp(Ek); dp(sskdict)
dp(sp.simplify(sp.expand(Ek)))

#dp(expr := sp.simplify(sp.expand(expr)))
dp(expr)
dp(exprs := sp.Matrix([expr[8:]]))


(3, 3)


Matrix([
[                         -{a_{m}}_{1}*{s_{a}}_{1} + {a_{p}}_{1}*{s_{a}}_{1} + {b_{p}}_{1}*{k_{x}}_{1}],
[                         -{a_{m}}_{4}*{s_{a}}_{2} + {a_{p}}_{1}*{s_{a}}_{2} + {b_{p}}_{1}*{k_{x}}_{2}],
[{a_{p}}_{1}*{s_{b}}_{1} - {b_{m}}_{1}*{s_{a}}_{1} - {b_{m}}_{2}*{s_{a}}_{2} + {b_{p}}_{1}*{k_{u}}_{1}],
[                         -{a_{m}}_{1}*{s_{a}}_{3} + {a_{p}}_{4}*{s_{a}}_{3} + {b_{p}}_{2}*{k_{x}}_{1}],
[                         -{a_{m}}_{4}*{s_{a}}_{4} + {a_{p}}_{4}*{s_{a}}_{4} + {b_{p}}_{2}*{k_{x}}_{2}],
[{a_{p}}_{4}*{s_{b}}_{2} - {b_{m}}_{1}*{s_{a}}_{3} - {b_{m}}_{2}*{s_{a}}_{4} + {b_{p}}_{2}*{k_{u}}_{1}],
[                                                                        {s_{a}}_{1} + {s_{a}}_{3} - 1],
[                                                                        {s_{a}}_{2} + {s_{a}}_{4} - 1],
[                                                                            {s_{b}}_{1} + {s_{b}}_{2}]])

(Matrix([
[-{a_{m}}_{1} + {a_{p}}_{1},                          0,                          0,                          0,           0,           0],
[                         0, -{a_{m}}_{4} + {a_{p}}_{1},                          0,                          0,           0,           0],
[              -{b_{m}}_{1},               -{b_{m}}_{2},                          0,                          0, {a_{p}}_{1},           0],
[                         0,                          0, -{a_{m}}_{1} + {a_{p}}_{4},                          0,           0,           0],
[                         0,                          0,                          0, -{a_{m}}_{4} + {a_{p}}_{4},           0,           0],
[                         0,                          0,               -{b_{m}}_{1},               -{b_{m}}_{2},           0, {a_{p}}_{4}],
[                         1,                          0,                          1,                          0,           0,           0],
[         

Matrix([
[1, 0, 0, 0, 0, 0, -({a_{p}}_{1}/{b_{m}}_{1} - {b_{m}}_{2}*({a_{m}}_{1}*{a_{p}}_{1} - {a_{p}}_{1}*{a_{p}}_{4})/({b_{m}}_{1}*({a_{m}}_{1}*{b_{m}}_{2} - {a_{p}}_{4}*{b_{m}}_{2})))*({b_{m}}_{1}**2 + {b_{m}}_{1}*{b_{m}}_{2} - {b_{m}}_{1}*{b_{p}}_{1}*{k_{u}}_{1} - {b_{m}}_{1}*{b_{p}}_{2}*{k_{u}}_{1})/(-{a_{p}}_{1}*{b_{m}}_{1} + {a_{p}}_{4}*{b_{m}}_{1}) + 1 + {b_{m}}_{2}*({a_{m}}_{1}*{b_{m}}_{1} + {a_{m}}_{1}*{b_{m}}_{2} - {a_{m}}_{1}*{b_{p}}_{1}*{k_{u}}_{1} - {a_{p}}_{4}*{b_{m}}_{1} - {a_{p}}_{4}*{b_{m}}_{2} + {a_{p}}_{4}*{b_{p}}_{1}*{k_{u}}_{1} - {b_{m}}_{1}*{b_{p}}_{2}*{k_{x}}_{1})/({b_{m}}_{1}*({a_{m}}_{1}*{b_{m}}_{2} - {a_{p}}_{4}*{b_{m}}_{2})) - ({b_{m}}_{1} + {b_{m}}_{2} - {b_{p}}_{1}*{k_{u}}_{1})/{b_{m}}_{1}],
[0, 1, 0, 0, 0, 0,                                                                                                                                                   -({a_{m}}_{1}*{a_{p}}_{1} - {a_{p}}_{1}*{a_{p}}_{4})*({b_{m}}_{1}**2 + {b_{m}}_{1}*{b_{m}}_{2} - {b_{m}}

{{s_{a}}_{1}: -({a_{p}}_{1}/{b_{m}}_{1} - {b_{m}}_{2}*({a_{m}}_{1}*{a_{p}}_{1} - {a_{p}}_{1}*{a_{p}}_{4})/({b_{m}}_{1}*({a_{m}}_{1}*{b_{m}}_{2} - {a_{p}}_{4}*{b_{m}}_{2})))*({b_{m}}_{1}**2 + {b_{m}}_{1}*{b_{m}}_{2} - {b_{m}}_{1}*{b_{p}}_{1}*{k_{u}}_{1} - {b_{m}}_{1}*{b_{p}}_{2}*{k_{u}}_{1})/(-{a_{p}}_{1}*{b_{m}}_{1} + {a_{p}}_{4}*{b_{m}}_{1}) + 1 + {b_{m}}_{2}*({a_{m}}_{1}*{b_{m}}_{1} + {a_{m}}_{1}*{b_{m}}_{2} - {a_{m}}_{1}*{b_{p}}_{1}*{k_{u}}_{1} - {a_{p}}_{4}*{b_{m}}_{1} - {a_{p}}_{4}*{b_{m}}_{2} + {a_{p}}_{4}*{b_{p}}_{1}*{k_{u}}_{1} - {b_{m}}_{1}*{b_{p}}_{2}*{k_{x}}_{1})/({b_{m}}_{1}*({a_{m}}_{1}*{b_{m}}_{2} - {a_{p}}_{4}*{b_{m}}_{2})) - ({b_{m}}_{1} + {b_{m}}_{2} - {b_{p}}_{1}*{k_{u}}_{1})/{b_{m}}_{1},
 {s_{a}}_{2}: -({a_{m}}_{1}*{a_{p}}_{1} - {a_{p}}_{1}*{a_{p}}_{4})*({b_{m}}_{1}**2 + {b_{m}}_{1}*{b_{m}}_{2} - {b_{m}}_{1}*{b_{p}}_{1}*{k_{u}}_{1} - {b_{m}}_{1}*{b_{p}}_{2}*{k_{u}}_{1})/(({a_{m}}_{1}*{b_{m}}_{2} - {a_{p}}_{4}*{b_{m}}_{2})*(-{a_{p}}_{1}*{b_{m}}_{1} + {a_{p}}_{4}*{b_{m

...

Matrix([
[1, 0, 0,                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

{{k_{x}}_{1}: -(-{a_{m}}_{1}**2 + {a_{m}}_{1}*{a_{p}}_{1} + {a_{m}}_{1}*{a_{p}}_{4} - {a_{p}}_{1}*{a_{p}}_{4})/({a_{m}}_{1}*{b_{p}}_{1} + {a_{m}}_{1}*{b_{p}}_{2} - {a_{p}}_{1}*{b_{p}}_{2} - {a_{p}}_{4}*{b_{p}}_{1}),
 {k_{x}}_{2}: -({a_{m}}_{1} - {a_{p}}_{4})*({a_{m}}_{1}*{a_{m}}_{4}*{a_{p}}_{1}*{b_{p}}_{1}*{b_{p}}_{2} + {a_{m}}_{1}*{a_{m}}_{4}*{a_{p}}_{1}*{b_{p}}_{2}**2 + {a_{m}}_{1}*{a_{m}}_{4}*{a_{p}}_{4}*{b_{p}}_{1}**2 + {a_{m}}_{1}*{a_{m}}_{4}*{a_{p}}_{4}*{b_{p}}_{1}*{b_{p}}_{2} - {a_{m}}_{1}*{a_{p}}_{1}**2*{b_{p}}_{1}*{b_{p}}_{2} - {a_{m}}_{1}*{a_{p}}_{1}**2*{b_{p}}_{2}**2 - {a_{m}}_{1}*{a_{p}}_{1}*{a_{p}}_{4}*{b_{p}}_{1}**2 - {a_{m}}_{1}*{a_{p}}_{1}*{a_{p}}_{4}*{b_{p}}_{1}*{b_{p}}_{2} - {a_{m}}_{4}*{a_{p}}_{1}**2*{b_{p}}_{2}**2 - 2*{a_{m}}_{4}*{a_{p}}_{1}*{a_{p}}_{4}*{b_{p}}_{1}*{b_{p}}_{2} - {a_{m}}_{4}*{a_{p}}_{4}**2*{b_{p}}_{1}**2 + {a_{p}}_{1}**3*{b_{p}}_{2}**2 + 2*{a_{p}}_{1}**2*{a_{p}}_{4}*{b_{p}}_{1}*{b_{p}}_{2} + {a_{p}}_{1}*{a_{p}}_{4}**2*{b_{p}}_{1}**2)*(-{a_{m}}_{1}**2

Matrix([
[1, 0, 0,                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

Matrix(1, 0, [])

In [79]:
#dp(sp.simplify(expr))
dp(exprs)
dp(ex1 := sp.Matrix([exprs[:4]]))
dp(sp.simplify(ex1))
dp(exprs := sp.Matrix([exprs[4:]]))
dp(kxs := sp.Matrix([exprs[:-1]])); dp(kus := sp.Matrix([exprs[-1]]))

Matrix(1, 0, [])

Matrix(1, 0, [])

Matrix(1, 0, [])

Matrix(1, 0, [])

Matrix(1, 0, [])

IndexError: list index out of range

In [ ]:
display(sp.Dict(ddict))

{NdimRdim: (3, 3), cgt: (Matrix([
[{a_{p}}_{1},           0,           0, 1],
[          0, {a_{p}}_{5},           0, 1],
[          0,           0, {a_{p}}_{9}, 1],
[{c_{p}}_{1}, {c_{p}}_{2}, {c_{p}}_{3}, 0]]), Matrix([
[{a_{m}}_{1},           0,           0, 1],
[          0, {a_{m}}_{5},           0, 1],
[          0,           0, {a_{m}}_{9}, 1],
[{c_{m}}_{1}, {c_{m}}_{2}, {c_{m}}_{3}, 0]]), Matrix([
[{s_{a}}_{1}, {s_{a}}_{2}, {s_{a}}_{3}, {s_{b}}_{1}],
[{s_{a}}_{4}, {s_{a}}_{5}, {s_{a}}_{6}, {s_{b}}_{2}],
[{s_{a}}_{7}, {s_{a}}_{8}, {s_{a}}_{9}, {s_{b}}_{3}],
[{k_{x}}_{1}, {k_{x}}_{2}, {k_{x}}_{3}, {k_{u}}_{1}]])), cgtsyms: (Matrix([
[{s_{a}}_{1}, {s_{a}}_{2}, {s_{a}}_{3}],
[{s_{a}}_{4}, {s_{a}}_{5}, {s_{a}}_{6}],
[{s_{a}}_{7}, {s_{a}}_{8}, {s_{a}}_{9}]]), Matrix([
[{s_{b}}_{1}],
[{s_{b}}_{2}],
[{s_{b}}_{3}]]), Matrix([[{k_{x}}_{1}, {k_{x}}_{2}, {k_{x}}_{3}]]), Matrix([[{k_{u}}_{1}]])), plant: (Matrix([
[{a_{p}}_{1}, {a_{p}}_{2}, {a_{p}}_{3}],
[{a_{p}}_{4}, {a_{p}}_{5}, {a_{p}}_{6}

In [ ]:
ddict.update([("kxkueq", sp.srepr(sp.Tuple(kxs, kus)))])
display(ff :=sp.Dict(ddict))
print(sp.srepr(ff))

{NdimRdim: (3, 3), cgt: (Matrix([
[{a_{p}}_{1},           0,           0, 1],
[          0, {a_{p}}_{5},           0, 1],
[          0,           0, {a_{p}}_{9}, 1],
[{c_{p}}_{1}, {c_{p}}_{2}, {c_{p}}_{3}, 0]]), Matrix([
[{a_{m}}_{1},           0,           0, 1],
[          0, {a_{m}}_{5},           0, 1],
[          0,           0, {a_{m}}_{9}, 1],
[{c_{m}}_{1}, {c_{m}}_{2}, {c_{m}}_{3}, 0]]), Matrix([
[{s_{a}}_{1}, {s_{a}}_{2}, {s_{a}}_{3}, {s_{b}}_{1}],
[{s_{a}}_{4}, {s_{a}}_{5}, {s_{a}}_{6}, {s_{b}}_{2}],
[{s_{a}}_{7}, {s_{a}}_{8}, {s_{a}}_{9}, {s_{b}}_{3}],
[{k_{x}}_{1}, {k_{x}}_{2}, {k_{x}}_{3}, {k_{u}}_{1}]])), cgtsyms: (Matrix([
[{s_{a}}_{1}, {s_{a}}_{2}, {s_{a}}_{3}],
[{s_{a}}_{4}, {s_{a}}_{5}, {s_{a}}_{6}],
[{s_{a}}_{7}, {s_{a}}_{8}, {s_{a}}_{9}]]), Matrix([
[{s_{b}}_{1}],
[{s_{b}}_{2}],
[{s_{b}}_{3}]]), Matrix([[{k_{x}}_{1}, {k_{x}}_{2}, {k_{x}}_{3}]]), Matrix([[{k_{u}}_{1}]])), kxkueq: (Matrix([[({c_{p}}_{1}*{k_{x}}_{1}*({a_{m}}_{1} - {a_{p}}_{5}) + ({a_{m}}_{1} - {a_{p}}_

Dict(Tuple(Symbol('NdimRdim'), Tuple(Integer(3), Integer(3))), Tuple(Symbol('plant'), Tuple(ImmutableDenseMatrix([[Symbol('{a_{p}}_{1}'), Symbol('{a_{p}}_{2}'), Symbol('{a_{p}}_{3}')], [Symbol('{a_{p}}_{4}'), Symbol('{a_{p}}_{5}'), Symbol('{a_{p}}_{6}')], [Symbol('{a_{p}}_{7}'), Symbol('{a_{p}}_{8}'), Symbol('{a_{p}}_{9}')]]), ImmutableDenseMatrix([[Symbol('{b_{p}}_{1}')], [Symbol('{b_{p}}_{2}')], [Symbol('{b_{p}}_{3}')]]), ImmutableDenseMatrix([[Symbol('{c_{p}}_{1}'), Symbol('{c_{p}}_{2}'), Symbol('{c_{p}}_{3}')]]))), Tuple(Symbol('refmodel'), Tuple(ImmutableDenseMatrix([[Symbol('{a_{m}}_{1}'), Symbol('{a_{m}}_{2}'), Symbol('{a_{m}}_{3}')], [Symbol('{a_{m}}_{4}'), Symbol('{a_{m}}_{5}'), Symbol('{a_{m}}_{6}')], [Symbol('{a_{m}}_{7}'), Symbol('{a_{m}}_{8}'), Symbol('{a_{m}}_{9}')]]), ImmutableDenseMatrix([[Symbol('{b_{m}}_{1}')], [Symbol('{b_{m}}_{2}')], [Symbol('{b_{m}}_{3}')]]), ImmutableDenseMatrix([[Symbol('{c_{m}}_{1}'), Symbol('{c_{m}}_{2}'), Symbol('{c_{m}}_{3}')]]))), Tuple(Symb

In [ ]:
# kx,kuについて解く
dp(exprs)
vals = [kx[0], kx[1], kx[2], ku[0], 1]
# 行列形に変形する
kxkusol, kmat, kval, ky = sct.eqs_to_mateqs(list(exprs), vals)
dp(kxkusol)
kxkxku1 = sp.simplify(sp.expand(kmat.inv()*ky))
dp(sp.Eq(kval, kxkxku1))
# 連立方程式として直接解く
rtt = sp.solve([sp.Eq(i, 0) for i in exprs], list(kval))
dp(kxs := sp.Matrix([ rtt[skey] for skey in vals[:-1] ]), "kxs:", pmode=5)

dp(sp.simplify(sp.expand(kxkxku1 - kxs)))
display(12)
print(sp.srepr(kxs))

Matrix([[({c_{p}}_{1}*{k_{x}}_{1}*({a_{m}}_{1} - {a_{p}}_{5}) + ({a_{m}}_{1} - {a_{p}}_{1})*({c_{p}}_{2}*{k_{x}}_{1} + ({a_{m}}_{1} - {a_{p}}_{5})*(-{c_{m}}_{1} + {c_{p}}_{3}*{k_{x}}_{1}/({a_{m}}_{1} - {a_{p}}_{9}))))/(({a_{m}}_{1} - {a_{p}}_{1})*({a_{m}}_{1} - {a_{p}}_{5})), ({c_{p}}_{1}*{k_{x}}_{2}*({a_{m}}_{5} - {a_{p}}_{5}) + ({a_{m}}_{5} - {a_{p}}_{1})*({c_{p}}_{2}*{k_{x}}_{2} + ({a_{m}}_{5} - {a_{p}}_{5})*(-{c_{m}}_{2} + {c_{p}}_{3}*{k_{x}}_{2}/({a_{m}}_{5} - {a_{p}}_{9}))))/(({a_{m}}_{5} - {a_{p}}_{1})*({a_{m}}_{5} - {a_{p}}_{5})), ({c_{p}}_{1}*{k_{x}}_{3}*({a_{m}}_{9} - {a_{p}}_{5}) + ({a_{m}}_{9} - {a_{p}}_{1})*({c_{p}}_{2}*{k_{x}}_{3} + ({a_{m}}_{9} - {a_{p}}_{5})*(-{c_{m}}_{3} + {c_{p}}_{3}*{k_{x}}_{3}/({a_{m}}_{9} - {a_{p}}_{9}))))/(({a_{m}}_{9} - {a_{p}}_{1})*({a_{m}}_{9} - {a_{p}}_{5})), ({a_{p}}_{1}*{a_{p}}_{9}*{c_{p}}_{2}*({a_{m}}_{1} - {a_{p}}_{1})*({a_{m}}_{5} - {a_{p}}_{1})*({a_{m}}_{9} - {a_{p}}_{1})*(-{k_{u}}_{1}*({a_{m}}_{1} - {a_{p}}_{5})*({a_{m}}_{5} - {a_{p}}_{

$$\left[\begin{matrix}\frac{{c_{p}}_{1} {k_{x}}_{1} \left({a_{m}}_{1} - {a_{p}}_{5}\right) + \left({a_{m}}_{1} - {a_{p}}_{1}\right) \left({c_{p}}_{2} {k_{x}}_{1} + \left({a_{m}}_{1} - {a_{p}}_{5}\right) \left(- {c_{m}}_{1} + \frac{{c_{p}}_{3} {k_{x}}_{1}}{{a_{m}}_{1} - {a_{p}}_{9}}\right)\right)}{\left({a_{m}}_{1} - {a_{p}}_{1}\right) \left({a_{m}}_{1} - {a_{p}}_{5}\right)} & \frac{{c_{p}}_{1} {k_{x}}_{2} \left({a_{m}}_{5} - {a_{p}}_{5}\right) + \left({a_{m}}_{5} - {a_{p}}_{1}\right) \left({c_{p}}_{2} {k_{x}}_{2} + \left({a_{m}}_{5} - {a_{p}}_{5}\right) \left(- {c_{m}}_{2} + \frac{{c_{p}}_{3} {k_{x}}_{2}}{{a_{m}}_{5} - {a_{p}}_{9}}\right)\right)}{\left({a_{m}}_{5} - {a_{p}}_{1}\right) \left({a_{m}}_{5} - {a_{p}}_{5}\right)} & \frac{{c_{p}}_{1} {k_{x}}_{3} \left({a_{m}}_{9} - {a_{p}}_{5}\right) + \left({a_{m}}_{9} - {a_{p}}_{1}\right) \left({c_{p}}_{2} {k_{x}}_{3} + \left({a_{m}}_{9} - {a_{p}}_{5}\right) \left(- {c_{m}}_{3} + \frac{{c_{p}}_{3} {k_{x}}_{3}}{{a_{m}}_{9} - {a_{p}}_{9}}\rig

In [ ]:
dp(exprs2 := sp.Matrix([exprs[0], exprs[3], exprs[2], exprs[1]]))
dp(exprs2 := sp.Matrix([exprs2[0], exprs2[1]-exprs2[2]*am[8], exprs2[2], exprs2[3]]))
vals = [ap[2], ap[5], ap[8], bp[0], bp[1], bp[2], 1]
cgtmat, smat, smatx, smaty = sct.eqs_to_mateqs(list(exprs2), vals)
dp(cgtmat, "同定式", pmode=5)

In [ ]:
# bpの推定式の導出のため2つの規範モデル用変数を準備
am2_1, am4_1, bm1_1, bm2_1, kx1_1, kx2_1, ku1_1 = sp.symbols("{a_{m}}_{2;1} {a_{m}}_{4;1} {b_{m}}_{1;1} {b_{m}}_{2;1} {k_{x}}_{1;1} {k_{x}}_{2;1} {k_{u}}_{1;1}")
am2_2, am4_2, bm1_2, bm2_2, kx1_2, kx2_2, ku1_2 = sp.symbols("{a_{m}}_{2;2} {a_{m}}_{4;2} {b_{m}}_{1;2} {b_{m}}_{2;2} {k_{x}}_{1;2} {k_{x}}_{2;2} {k_{u}}_{1;2}")
knownvals = sp.Matrix([[am2_1, am4_1, bm1_1, bm2_1, kx1_1, kx2_1, ku1_1], [am2_2, am4_2, bm1_2, bm2_2, kx1_2, kx2_2, ku1_2]])
dp(knownvals, "knownvals", pmode=5)

In [ ]:
allparamdict = {"ap": ap, "bp": bp, "cp": cp, "am": am, "bm": bm, "cm": cm, "kx": kx, "ku": ku,
                "am_1": (am2_1, am4_1), "bm_1": (bm1_1, bm2_1), "bm_2": (bm1_2, bm2_2), "kx_1": (kx1_1, kx2_1), "ku_1": (ku1_1,), 
                "am_2": (am2_2, am4_2),  "kx_2": (kx1_2, kx2_2), "ku_2": (ku1_2,)}
dp(allparamdict)
display(sp.srepr(allparamdict))

In [ ]:
lfunc = sp.parse_expr("MutableDenseMatrix([[Add(Mul(Integer(-1), Symbol('{a_{m}}_{2}'), Add(Mul(Integer(-1), Symbol('{b_{p}}_{2}'), Symbol('{k_{x}}_{1}')), Integer(1))), Mul(Integer(-1), Symbol('{a_{m}}_{4}'), Symbol('{b_{p}}_{1}'), Symbol('{k_{x}}_{1}')), Symbol('{a_{p}}_{2}'), Mul(Symbol('{b_{p}}_{1}'), Symbol('{k_{x}}_{2}')))], [Add(Mul(Integer(-1), Symbol('{b_{m}}_{1}'), Add(Mul(Integer(-1), Symbol('{b_{p}}_{2}'), Symbol('{k_{x}}_{1}')), Integer(1))), Mul(Integer(-1), Symbol('{b_{m}}_{2}'), Symbol('{b_{p}}_{1}'), Symbol('{k_{x}}_{1}')), Mul(Symbol('{b_{p}}_{1}'), Symbol('{k_{u}}_{1}')))], [Add(Mul(Integer(-1), Symbol('{a_{m}}_{4}')), Symbol('{a_{p}}_{4}'), Mul(Symbol('{b_{p}}_{1}'), Symbol('{k_{x}}_{1}')), Mul(Symbol('{b_{p}}_{2}'), Symbol('{k_{x}}_{2}')))]])")
display(lfunc)
unparams = sp.Matrix([ap[1], ap[3]])
dp(unparams)
anss1 = sp.solve(lfunc, unparams)
anss2 = sp.simplify(sp.Matrix([ [ anss1[ap[1]] ], [ anss1[ap[3]] ] ]))
dp(anss2)

In [ ]:


def get_ss(num, den):
    ss_sys = ct.tf2ss(ct.tf(num, den), method="scipy")#scipyでないとクラッシュ
    T = np.array([[0, 1],[1, 0]])#座標変換行列
    ss_sys = ct.similarity_transform(ss_sys,T)
    return ss_sys

# シンボリック行列である成分, 変数s, システムの次元(Aのサイズ)n
def calc_pz(params, n):
    fnp()
    s = sp.symbols('s')
    if n == 2:
        A = sp.Matrix([[0, params[0]], [1, params[1]]])
        B = sp.Matrix([[params[2], params[3]]]).transpose()
        C = sp.Matrix([[0, 1]])
    if n != 2:
        raise ValueError("n must be 2")
    tf = C * (s*sp.eye(n) - A).inv() * B
    dp(sp.latex(A)+sp.latex(B)+sp.latex(C), "ABC:", pmode=6)
    dp(tf := sp.simplify(sp.expand(tf[0,0])) , "伝達関数:", pmode=5)
    nu = sp.numer(tf); de  = sp.denom(tf)
    dp(poze := ((sp.solve(de)[0]).evalf(), (sp.solve(de)[1]).evalf(), sp.solve(nu)), "poles, zeros:", pmode=5)
    return tf, poze


